# CW_09
# Phase Conjugation in Non-Hermitan Systems

In this exercise, you will demonstrate that phase conjugation does not work in a non-Hermitian system (with complex eigenvalues). To do this, you will show that propagating through a multimode fiber that has gain or loss breaks the methodology.

In [11]:
# - No modification necessary -

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets.widgets import interact
import ipywidgets as widgets

In [12]:
# - No modification necessary -

# Global simulation parameters
Nx = 256
Ny = 256
Lx = 100e-6
Ly = 100e-6

dx = Lx / Nx
dy = Ly / Ny

wavelength = 1.55e-6
k0 = 2 * np.pi / wavelength

core_radius = 25e-6
n_core = 1.45
n_clad = 1.44

dz = 1e-5
n_steps = 200

# Coordinate grids
x = np.linspace(-Lx/2, Lx/2, Nx)
y = np.linspace(-Ly/2, Ly/2, Ny)
X, Y = np.meshgrid(x, y)

# Frequency grids
fx = np.fft.fftfreq(Nx, dx)
fy = np.fft.fftfreq(Ny, dy)
FX, FY = np.meshgrid(fx, fy)
KX = 2 * np.pi * FX
KY = 2 * np.pi * FY

K2 = KX**2 + KY**2

In [13]:
# - No modification necessary -

def plot_field(field, title="Field"):
    intensity = np.abs(field)**2
    phase = np.angle(field)

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))

    im0 = axs[0].imshow(intensity, cmap='inferno')
    axs[0].set_title(f"{title} - Intensity")
    plt.colorbar(im0, ax=axs[0])

    im1 = axs[1].imshow(phase, cmap='twilight')
    axs[1].set_title(f"{title} - Phase")
    plt.colorbar(im1, ax=axs[1])

    plt.tight_layout()
    plt.show()

# Part 1

In this part, you will need to create your fiber and give it gain or loss through the complex index of refraction. 

To do this, you will implement the function create_fiber_index, which takes the gain in the core region and the gain in the cladding region as parameters.

You will need to first build the real refractive index profile, then add on the imaginary component to each region independently based on their respective gain coefficients. Be sure to convert the provided gain coefficient (in units of 1/m) into an extinction coefficient (the actual imaginary part of the refractive index).

Once you have done this, you can use the provided visualization function to check your results. Then, answer the following questions:
1) How does the gain coefficient relate to the extinction coefficient? What does each part of the conversion represent (i.e. how does this conversion really work)?
2) For this exercise, we use gain coefficients in between $\pm$2000 [1/m], or $\pm$2 [1/mm]. What do you notice about the magnitude of the imaginary component of the refractive index in this regime relative to its real component.



In [14]:
def create_fiber_index(core_gain=0.0, cladding_gain=0.0):
    r = np.sqrt(X**2 + Y**2)

    # Real refractive index
    n_real = np.where(r < core_radius, n_core, n_clad)

    # Convert gain coefficient (g) → extinction coefficient (k)
    # k = -g / (2 * k0)
    core_k = -core_gain / (2 * k0)
    cladding_k = -cladding_gain / (2 * k0)

    # Imaginary part of refractive index
    n_imag = np.zeros_like(r)
    n_imag[r < core_radius] = core_k
    n_imag[r >= core_radius] = cladding_k

    n_complex = n_real + 1j * n_imag
    return n_complex

In [15]:
# - No modification necessary -

def plot_index(n_complex):
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))

    im0 = axs[0].imshow(np.real(n_complex))
    axs[0].set_title("Refractive Index (Real)")
    plt.colorbar(im0, ax=axs[0])

    im1 = axs[1].imshow(np.imag(n_complex))
    axs[1].set_title("Gain/Loss (Imaginary)")
    plt.colorbar(im1, ax=axs[1])

    plt.tight_layout()
    plt.show()
    
def visualize_refractive_index(core_gain, cladding_gain):
    print(f"Core Gain: {core_gain:.5f}")
    print(f"Cladding Gain: {cladding_gain:.5f}")
    
    n_complex = create_fiber_index(core_gain, cladding_gain)

    plot_index(n_complex)

interact(
    visualize_refractive_index,
    core_gain=widgets.FloatSlider(
        value=0.0,
        min=-2000,
        max=2000,
        step=50,
        description='Core Gain (1/m)',
        continuous_update=False
    ),
    cladding_gain=widgets.FloatSlider(
        value=0.0,
        min=-2000,
        max=2000,
        step=50,
        description='Cladding Gain (1/m)',
        continuous_update=False
    )
)

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='Core Gain (1/m)', max=2000.…

<function __main__.visualize_refractive_index(core_gain, cladding_gain)>

## Discussion
1) The conversion between the gain and the extinction coefficient is given as $\kappa = -\frac{g}{2k_0}$. The three components are explained as such. First, the negative sign comes from the fact that the gain coefficient being positive (i.e. growing signal) must correspond to a positive exponent after doing $e^{i(i\kappa)}$, so it must be switched to negative to cancel out the $i^2$. Dividing by 2 is necessary to convert from a gain that is defined in terms of the intensity of the field to a gain that affects the amplitude of the field. Lastly, dividing by $k_0$ is necessary to normalize the result so that $k_0$ can be multiplied in when calculating the phase shift, just as it is done for the real component.
2) Despite the fact that we actually have very large values of gain or loss, the actual values of the extinction coefficient are very small, varying from around -2e-4 to 2e-4. 

# Part 2

In this part, you will put together the remainder of the helper functions needed to test the phase conjugation methodology with gain/loss.

Here you should implement the functions generate_input_field, which will generate a gaussian input beam of a given waist size with quadratic phase of the same waist, and propagate, which will perform the beam propagation method through the complex index medium.

After implementing those functions, you will combine everything in the function run_simulation, which will take a core gain and a cladding gain and output the result of propagating, phase conjugating, and propagating again.

Then use the provided visualization code to examine the resulting effects. Answer the following questions:
1) Ignoring general scaling, which effect seems to be more detrimental to the phase conjugation method, gain or loss? Does this depend on the region, and if so, how do we explain that?
2) What do you notice when the difference between the core and cladding gain is held constant but their actual values are varied? How do you explain this?
3) What loss values are typical in commercial fibers? Over the length used in this simulation, would they act Hermitian or non-Hermitian? 

In [16]:
def generate_input_field(w0=10e-6):
    field = np.exp(-(X**2 + Y**2) / w0**2)

    # Add some phase structure (multimode-like)
    phase = np.exp(1j * -(X**2 + Y**2) / w0**2)

    return field * phase

In [17]:
def propagate(field, n_complex, n_ref=1.44):
    field_z = field.copy()

    # Precompute diffraction operator (constant)
    H = np.exp(-1j * K2 * dz / (2 * k0 * n_ref))

    for _ in range(n_steps):
        # --- Diffraction step (Fourier domain) ---
        field_k = np.fft.fft2(field_z)
        field_k *= H
        field_z = np.fft.ifft2(field_k)

        # --- Index perturbation step (real domain) ---
        delta_n = n_complex - n_ref
        phase_shift = np.exp(1j * k0 * delta_n * dz)

        field_z *= phase_shift

    return field_z

In [18]:
def run_simulation(core_gain, cladding_gain):
    # Fiber
    n_complex = create_fiber_index(core_gain, cladding_gain)

    # Input field
    input_field = generate_input_field()

    # Forward propagation
    output_forward = propagate(input_field, n_complex)

    # Phase conjugation
    conjugated = np.conj(output_forward)

    # Backward propagation
    output_backward = propagate(conjugated, n_complex)

    return input_field, output_forward, output_backward, n_complex

In [19]:
# - No modification necessary -

def compute_error(input_field, output_field):
    input_intensity = np.abs(input_field)**2
    output_intensity = np.abs(output_field)**2

    mse = np.mean((input_intensity - output_intensity)**2)
    return mse

def interactive_simulation(core_gain, cladding_gain):
    plt.close('all')

    input_field, forward, backward, n_complex = run_simulation(core_gain, cladding_gain)

    print(f"Core Gain: {core_gain:.5f}")
    print(f"Cladding Gain: {cladding_gain:.5f}")

    plot_field(input_field, "Input Field")
    plot_field(backward, "After Phase Conjugation + Backward Propagation")

    error = compute_error(input_field, backward)
    print(f"Reconstruction Error: {error:.6f}")
    
interact(
    interactive_simulation,
    core_gain=widgets.FloatSlider(
        value=0.0,
        min=-2000,
        max=2000,
        step=50,
        description='Core Gain (1/m)',
        continuous_update=False
    ),
    cladding_gain=widgets.FloatSlider(
        value=0.0,
        min=-2000,
        max=2000,
        step=50,
        description='Cladding Gain (1/m)',
        continuous_update=False
    )
)

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='Core Gain (1/m)', max=2000.…

<function __main__.interactive_simulation(core_gain, cladding_gain)>

## Discussion
1) Inside of the core, loss seems to be the more detrimental effect to the shape of the returned field. In the cladding, gain seems to be more impactful. From this, we can deduce that relative gain in the cladding, which amplifies our higher order and escaping modes, creates the largest detrimental effect after phase conjugation.
2) When the delta between the core and cladding gain coefficients is kept constant but their values are varied, the actual resulting intensity pattern does not change other than in amplitude.. This makes intuitive sense because we can think about the larger shift as a general loss or gain applied on top of the actual structure of the fiber (just like we do with a reference refractive index), and we do not expect that background level of gain to cause any effect other than a global scaling.
3) A typical commercial fiber has a net loss of just a few dB per kilometer. After accounting for converting into the linear gain coefficient (by multiplying by 4.343), this is a coefficient of around 10 [1/km]. We are only propagating 2 mm, so at this length the loss we experience is negligible. This means that if we were using a real fiber we could approximate it as Hermitian. 